In [1]:
# 1. Import Libraries

import pandas as pd
import numpy as np
import ast
import random

pd.set_option('future.no_silent_downcasting', True)

from difflib import get_close_matches

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# 2. Load Ratings Dataset

ratings = pd.read_csv(
    "data/tmdb_movie_ratings.csv"
)

# Limit users
ratings = ratings[
    ratings["userId"] <= 600
].copy()

print("Ratings Shape:", ratings.shape)

display(ratings.head())

Ratings Shape: (55511, 4)


,userId,ratingId,rating,timestamp
21362,312,1,4.0,2011-08-22 17:35:35
21363,304,1,4.0,1999-10-04 11:25:43
21364,302,1,5.0,2016-08-10 19:19:36
21365,301,1,3.0,2017-01-17 18:11:16
21366,298,1,5.0,1997-02-10 15:23:28


In [3]:
# 3. Load Movie Dataset

movies = pd.read_csv(
    "data/tmdb_movie_dataset.csv",
    usecols=[
        "ratingId",
        "tmdbId",
        "title",
        "vote_average",
        "vote_count"
    ]
)

print("Movie Dataset Shape:", movies.shape)

display(movies.head())

Movie Dataset Shape: (4602, 5)


,tmdbId,title,vote_average,vote_count,ratingId
0,5,Four Rooms,6.5,530,18
1,11,Star Wars,8.1,6624,260
2,12,Finding Nemo,7.6,6122,6377
3,13,Forrest Gump,8.2,7927,356
4,14,American Beauty,7.9,3313,2858


In [4]:
# 4. Check Columns

print("Ratings Columns:")
print(ratings.columns.tolist())

print("\nMovie Dataset Columns:")
print(movies.columns.tolist())

Ratings Columns:
['userId', 'ratingId', 'rating', 'timestamp']

Movie Dataset Columns:
['tmdbId', 'title', 'vote_average', 'vote_count', 'ratingId']


In [5]:
# 5. Merge Ratings with Movie Dataset

merged_data = pd.merge(
    ratings,
    movies,
    on="ratingId",
    how="inner"
)

print("Merged Dataset:")

display(
    merged_data[
        [
            "userId",
            "ratingId",
            "rating",
            "tmdbId",
            "title",
            "vote_average",
            "vote_count"
        ]
    ].head()
)

print("Shape:", merged_data.shape)

Merged Dataset:


,userId,ratingId,rating,tmdbId,title,vote_average,vote_count
0,312,1,4.0,862,Toy Story,7.7,5269
1,304,1,4.0,862,Toy Story,7.7,5269
2,302,1,5.0,862,Toy Story,7.7,5269
3,301,1,3.0,862,Toy Story,7.7,5269
4,298,1,5.0,862,Toy Story,7.7,5269


Shape: (55511, 8)


In [6]:
# 6. Create Collaborative Filtering Dataset

cf_data = merged_data[
    [
        "userId",
        "ratingId",
        "tmdbId",
        "title",
        "rating"
    ]
].copy()

print("Collaborative Filtering Dataset:")

display(cf_data.head())

print("Shape:", cf_data.shape)

Collaborative Filtering Dataset:


,userId,ratingId,tmdbId,title,rating
0,312,1,862,Toy Story,4.0
1,304,1,862,Toy Story,4.0
2,302,1,862,Toy Story,5.0
3,301,1,862,Toy Story,3.0
4,298,1,862,Toy Story,5.0


Shape: (55511, 5)


In [7]:
# 7. Movie Statistics

movie_stats = pd.DataFrame(
    cf_data.groupby("title")["rating"].mean()
)

movie_stats["number of ratings"] = (
    cf_data.groupby("title")["rating"].count()
)

display(movie_stats.head())

,rating,number of ratings
title,,
(500) Days of Summer,3.900000,40
10 Cloverfield Lane,4.000000,12
10 Things I Hate About You,3.846939,49
102 Dalmatians,1.750000,6
11:14,3.166667,3


In [8]:
# 8. Movies with Most User Ratings

top_rated_movies = (
    movie_stats
    .sort_values(
        "number of ratings",
        ascending=False
    )
    .head(10)
)

display(top_rated_movies)

,rating,number of ratings
title,,
Forrest Gump,4.074074,297
Pulp Fiction,4.137324,284
The Shawshank Redemption,4.474820,278
The Silence of the Lambs,4.207692,260
The Matrix,4.179134,254
Jurassic Park,3.702083,240
Star Wars,4.188841,233
Schindler's List,4.375000,228
Braveheart,4.102439,205


In [9]:
# 9. Create User-Movie Matrix

movie_matrix = cf_data.pivot_table(
    index="userId",
    columns="title",
    values="rating"
)

display(movie_matrix.head())

print(
    "User-Movie Matrix Shape:",
    movie_matrix.shape
)

title,(500) Days of Summer,10 Cloverfield Lane,10 Things I Hate About You,102 Dalmatians,11:14,12 Angry Men,12 Rounds,12 Years a Slave,127 Hours,13 Going on 30,...,Zombieland,Zookeeper,Zoolander,Zoolander 2,[REC],[REC]²,eXistenZ,xXx,xXx: State of the Union,Æon Flux
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,4.0,NaN,3.5,NaN,NaN,NaN,NaN,3.5,NaN,4.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


User-Movie Matrix Shape: (600, 3439)


In [10]:
# 10. Load Full TMDB Movie Dataset for Content-Based Filtering

df = pd.read_csv(
    "data/tmdb_movie_dataset.csv"
)

print("Dataset Shape:", df.shape)

print("Columns:")
print(df.columns.tolist())

display(
    df[
        [
            "tmdbId",
            "title",
            "release_date",
            "vote_average"
        ]
    ].head(3)
)

Dataset Shape: (4602, 21)
Columns:
['budget', 'genres', 'homepage', 'tmdbId', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'ratingId']


,tmdbId,title,release_date,vote_average
0,5,Four Rooms,1995-12-09,6.5
1,11,Star Wars,1977-05-25,8.1
2,12,Finding Nemo,2003-05-30,7.6


In [11]:
# 11. Data Pre-processing and Feature Extraction

def extract_names(text):

    try:

        items = ast.literal_eval(text)

        if isinstance(items, list):

            return " ".join(
                str(item.get("name", ""))
                for item in items
                if isinstance(item, dict)
            )

    except (
        ValueError,
        SyntaxError,
        TypeError
    ):

        pass

    return ""


# Convert genres

df["genres_clean"] = (
    df["genres"]
    .fillna("")
    .apply(extract_names)
)


# Convert keywords

df["keywords_clean"] = (
    df["keywords"]
    .fillna("")
    .apply(extract_names)
)


# Clean overview

df["overview"] = (
    df["overview"]
    .fillna("")
)


# Extract release year

df["year"] = (
    pd.to_datetime(
        df["release_date"],
        errors="coerce"
    )
    .dt.year
    .astype("Int64")
)


display(
    df[
        [
            "title",
            "genres_clean",
            "keywords_clean",
            "overview"
        ]
    ].head(3)
)

,title,genres_clean,keywords_clean,overview
0,Four Rooms,Crime Comedy,hotel new year's eve witch bet hotel room sper...,It's Ted the Bellhop's first night on the job....
1,Star Wars,Adventure Action Science Fiction,android galaxy hermit death star lightsaber je...,Princess Leia is captured and held hostage by ...
2,Finding Nemo,Animation Family,father son relationship harbor underwater fish...,"Nemo, an adventurous young clownfish, is unexp..."


In [12]:
# 12. Combine Movie Attributes

df["content"] = (
    df["genres_clean"] + " " +
    df["keywords_clean"] + " " +
    df["overview"]
).str.strip()


# Remove rows without title

df = df[
    df["title"].notna()
].copy()


df["content"] = (
    df["content"]
    .fillna("")
)


display(
    df[
        [
            "title",
            "genres_clean",
            "keywords_clean",
            "overview"
        ]
    ].head(3)
)

,title,genres_clean,keywords_clean,overview
0,Four Rooms,Crime Comedy,hotel new year's eve witch bet hotel room sper...,It's Ted the Bellhop's first night on the job....
1,Star Wars,Adventure Action Science Fiction,android galaxy hermit death star lightsaber je...,Princess Leia is captured and held hostage by ...
2,Finding Nemo,Animation Family,father son relationship harbor underwater fish...,"Nemo, an adventurous young clownfish, is unexp..."


In [13]:
# 13. Create TF-IDF Matrix

tfidf = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(
    df["content"]
)

print(
    "TF-IDF Matrix Size:",
    tfidf_matrix.shape
)

TF-IDF Matrix Size: (4602, 22383)


In [14]:
# 14. Create Cosine Similarity Matrix

cosine_sim = cosine_similarity(
    tfidf_matrix,
    tfidf_matrix
)

print(
    "Cosine Similarity Matrix Size:",
    cosine_sim.shape
)

Cosine Similarity Matrix Size: (4602, 4602)


In [15]:
# 15. Create Movie Title Index

indices = pd.Series(
    df.index,
    index=df["title"]
    .str.strip()
    .str.lower()
).drop_duplicates()

print(
    "Number of Movie Titles:",
    len(indices)
)

Number of Movie Titles: 4602


In [16]:
# 16. Find Movie Title

def find_movie_title(title):

    query = str(title).strip().lower()

    if not query:
        return None


    # Exact match

    if query in indices.index:

        return df.loc[
            indices[query],
            "title"
        ]


    # Partial match

    partial_matches = [

        original_title

        for original_title
        in df["title"].dropna().unique()

        if query in str(
            original_title
        ).lower()

    ]

    if partial_matches:

        return partial_matches[0]


    # Fuzzy match

    title_map = {

        str(t).lower(): str(t)

        for t
        in df["title"].dropna().unique()

    }


    close = get_close_matches(

        query,

        list(
            title_map.keys()
        ),

        n=1,

        cutoff=0.65

    )


    if close:

        return title_map[
            close[0]
        ]


    return None

In [17]:
# 17. Content-Based Recommendation Function

def recommend_content(
    title,
    top_n=10
):

    matched_title = find_movie_title(
        title
    )

    if matched_title is None:

        return None


    idx = indices[
        matched_title.lower()
    ]


    sim_scores = list(
        enumerate(
            cosine_sim[idx]
        )
    )


    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )


    # Remove selected movie

    sim_scores = sim_scores[
        1:top_n + 1
    ]


    movie_indices = [
        i
        for i, _
        in sim_scores
    ]


    scores = [
        round(score, 4)
        for _, score
        in sim_scores
    ]


    result = df[
        [
            "title",
            "year",
            "genres_clean",
            "keywords_clean"
        ]
    ].iloc[
        movie_indices
    ].copy()


    result.insert(
        0,
        "rank",
        range(
            1,
            len(result) + 1
        )
    )


    result["similarity_score"] = scores


    result["keywords_clean"] = (
        result["keywords_clean"]
        .apply(
            lambda x:
            ", ".join(
                x.split()[:5]
            )
            if x
            else ""
        )
    )


    result = result.rename(
        columns={
            "title": "movie_title",
            "genres_clean": "genres",
            "keywords_clean": "keywords"
        }
    )


    return result.reset_index(
        drop=True
    )

In [18]:
# 18. Hybrid Filtering Weights

CF_WEIGHT = 0.6

CONTENT_WEIGHT = 0.4


print(
    "Collaborative Filtering Weight:",
    CF_WEIGHT
)

print(
    "Content-Based Filtering Weight:",
    CONTENT_WEIGHT
)

Collaborative Filtering Weight: 0.6
Content-Based Filtering Weight: 0.4


In [19]:
# 19. Hybrid Recommendation Function

def hybrid_recommend(
    title,
    top_n=10
):

    # ==================================================
    # FIND MOVIE
    # ==================================================

    matched_title = find_movie_title(
        title
    )


    if matched_title is None:

        print(
            "Movie not found in dataset."
        )

        return None


    print(
        "Selected Movie:",
        matched_title
    )


    # ==================================================
    # CONTENT-BASED SCORE
    # ==================================================

    content_idx = indices[
        matched_title.lower()
    ]


    content_similarity = (
        cosine_sim[content_idx]
    )


    content_scores = pd.DataFrame({

        "title":
        df["title"].values,

        "Content Score":
        content_similarity

    })


    # Remove selected movie

    content_scores = content_scores[
        content_scores["title"]
        != matched_title
    ].copy()


    # Remove duplicate titles

    content_scores = (
        content_scores
        .drop_duplicates(
            subset=["title"]
        )
    )


    # ==================================================
    # COLLABORATIVE FILTERING SCORE
    # ==================================================

    cf_scores = pd.DataFrame(
        columns=[
            "title",
            "CF Score"
        ]
    )


    # Check whether movie exists in CF

    if matched_title in movie_matrix.columns:


        # ------------------------------------------------
        # IMPORTANT FIX
        # Only use movies with enough ratings
        # BEFORE calculating correlation.
        # ------------------------------------------------

        valid_cf_titles = (
            movie_stats[
                movie_stats[
                    "number of ratings"
                ] >= 100
            ]
            .index
        )


        # Keep only valid titles
        # that actually exist in matrix

        valid_cf_titles = (
            valid_cf_titles
            .intersection(
                movie_matrix.columns
            )
        )


        # Make sure selected movie is included

        if matched_title in valid_cf_titles:


            valid_movie_matrix = (
                movie_matrix[
                    valid_cf_titles
                ]
            )


            # ------------------------------------------------
            # Remove movies with zero variance
            # ------------------------------------------------

            rating_std = (
                valid_movie_matrix
                .std()
            )


            valid_columns = (
                rating_std[
                    rating_std > 0
                ]
                .index
            )


            # Selected movie must remain

            if matched_title in valid_columns:


                valid_movie_matrix = (
                    valid_movie_matrix[
                        valid_columns
                    ]
                )


                # ------------------------------------------------
                # Calculate correlation
                # ------------------------------------------------

                similar = (
                    valid_movie_matrix
                    .corrwith(
                        valid_movie_matrix[
                            matched_title
                        ]
                    )
                )


                # Convert to DataFrame

                temp_cf = pd.DataFrame({

                    "title":
                    similar.index,

                    "Correlation":
                    similar.values

                })


                # Remove NaN correlations

                temp_cf = temp_cf[
                    temp_cf[
                        "Correlation"
                    ].notna()
                ]


                # Remove selected movie

                temp_cf = temp_cf[
                    temp_cf["title"]
                    != matched_title
                ]


                # ------------------------------------------------
                # Normalize correlation
                # -1 to +1
                # becomes
                # 0 to 1
                # ------------------------------------------------

                temp_cf["CF Score"] = (
                    temp_cf["Correlation"] + 1
                ) / 2


                cf_scores = temp_cf[
                    [
                        "title",
                        "CF Score"
                    ]
                ]


    # ==================================================
    # COMBINE CONTENT + CF
    # ==================================================

    hybrid = pd.merge(

        content_scores,

        cf_scores,

        on="title",

        how="left"

    )


    # ==================================================
    # HANDLE MOVIES WITHOUT CF SCORE
    # ==================================================

    hybrid["CF Score"] = (
        hybrid["CF Score"]
        .infer_objects(copy=False).fillna(0)
    )


    hybrid["Content Score"] = (
        hybrid["Content Score"]
        .infer_objects(copy=False).fillna(0)
    )


    # ==================================================
    # HYBRID SCORE
    # ==================================================

    hybrid["Hybrid Score"] = (

        CF_WEIGHT *
        hybrid["CF Score"]

        +

        CONTENT_WEIGHT *
        hybrid["Content Score"]

    )
    # ==================================================
    # WHY THIS MOVIE?
    # ==================================================

    def generate_reason(row):

        cf = row["CF Score"]
        content = row["Content Score"]

        if cf >= 0.80 and content >= 0.80:
            return "Strong user-rating similarity + strong content similarity"

        elif cf >= 0.80:
            return "Users with similar rating behaviour also liked this movie"

        elif content >= 0.80:
            return "Strong similarity in genres, keywords and movie overview"

        elif cf >= 0.60 and content >= 0.60:
            return "Good balance of user-rating and content similarity"

        elif cf >= content:
            return "Recommended mainly from similar user-rating behaviour"

        else:
            return "Recommended mainly from similar movie content"

    hybrid["Why this movie?"] = hybrid.apply(
        generate_reason,
        axis=1
    )


    # ==================================================
    # IMPORTANT:
    # If CF data is unavailable for the selected movie,
    # use content-based recommendations instead.
    # ==================================================

    if cf_scores.empty:

        hybrid["Hybrid Score"] = (
            hybrid["Content Score"]
        )


    # ==================================================
    # SORT
    # ==================================================

    hybrid = hybrid.sort_values(

        "Hybrid Score",

        ascending=False

    )


    # ==================================================
    # TOP N
    # ==================================================

    hybrid = hybrid.head(
        top_n
    ).copy()


    # ==================================================
    # ADD MOVIE INFORMATION
    # ==================================================

    movie_information = df[
        [
            "title",
            "year",
            "genres_clean",
            "keywords_clean"
        ]
    ].drop_duplicates(
        subset=["title"]
    )


    hybrid = hybrid.merge(

        movie_information,

        on="title",

        how="left"

    )


    # ==================================================
    # FORMAT KEYWORDS
    # ==================================================

    hybrid["keywords_clean"] = (

        hybrid["keywords_clean"]
        .fillna("")
        .apply(

            lambda x:

            ", ".join(
                x.split()[:5]
            )

            if x

            else ""

        )

    )


    # ==================================================
    # RANK
    # ==================================================

    hybrid.insert(

        0,

        "Rank",

        range(
            1,
            len(hybrid) + 1
        )

    )


    # ==================================================
    # RENAME COLUMNS
    # ==================================================

    hybrid = hybrid.rename(

        columns={

            "title":
            "Movie Title",

            "year":
            "Year",

            "genres_clean":
            "Genres",

            "keywords_clean":
            "Keywords"

        }

    )


    # ==================================================
    # ROUND SCORES
    # ==================================================

    hybrid["CF Score"] = (
        hybrid["CF Score"]
        .round(4)
    )


    hybrid["Content Score"] = (
        hybrid["Content Score"]
        .round(4)
    )


    hybrid["Hybrid Score"] = (
        hybrid["Hybrid Score"]
        .round(4)
    )


    # ==================================================
    # FINAL OUTPUT
    # ==================================================

    return hybrid[
        [
            "Rank",
            "Movie Title",
            "Year",
            "Genres",
            "Keywords",
            "CF Score",
            "Content Score",
            "Hybrid Score",
            "Why this movie?"
        ]
    ].reset_index(
        drop=True
    )

In [20]:
# 20. Test Hybrid Recommendation

hybrid_result = hybrid_recommend(
    "Avatar",
    top_n=10
)

display(
    hybrid_result
)

Selected Movie: Avatar


,Rank,Movie Title,Year,Genres,Keywords,CF Score,Content Score,Hybrid Score,Why this movie?
0,1,Mission to Mars,2000,Science Fiction,"mars, spacecraft, space, travel, alien",0.0,0.3055,0.3055,Recommended mainly from similar movie content
1,2,Aliens,1986,Horror Action Thriller Science Fiction,"android, extraterrestrial, technology, space, ...",0.0,0.2876,0.2876,Recommended mainly from similar movie content
2,3,Moonraker,1979,Action Adventure Thriller Science Fiction,"venice, mass, murder, space, marine",0.0,0.2824,0.2824,Recommended mainly from similar movie content
3,4,Alien³,1992,Science Fiction Action Horror,"prison, android, spacecraft, space, marine",0.0,0.2766,0.2766,Recommended mainly from similar movie content
4,5,Spaceballs,1987,Comedy Science Fiction,"android, lasergun, swordplay, temple, space",0.0,0.2561,0.2561,Recommended mainly from similar movie content
5,6,Lifeforce,1985,Fantasy Horror Science Fiction Thriller,"space, marine, vampire, flying, saucer",0.0,0.2549,0.2549,Recommended mainly from similar movie content
6,7,Treasure Planet,2002,Adventure Animation Family Fantasy Science Fic...,"cyborg, based, on, novel, space",0.0,0.2532,0.2532,Recommended mainly from similar movie content
7,8,Lockout,2012,Action Thriller Science Fiction,"usa, president, anti, hero, dementia",0.0,0.2523,0.2523,Recommended mainly from similar movie content
8,9,Alien,1979,Horror Action Thriller Science Fiction,"android, countdown, space, marine, space",0.0,0.2457,0.2457,Recommended mainly from similar movie content
9,10,Planet of the Apes,2001,Thriller Science Fiction Action Adventure,"gorilla, space, marine, space, suit",0.0,0.2453,0.2453,Recommended mainly from similar movie content


In [21]:
# 21. User Input

favorite_movie = input(
    "Please enter your favorite movie: "
).strip()


hybrid_result = hybrid_recommend(
    favorite_movie,
    top_n=10
)


if hybrid_result is not None:

    print(
        "\nTop 10 Hybrid Recommendations:"
    )

    display(
        hybrid_result
    )

Please enter your favorite movie:  Avatar


Selected Movie: Avatar

Top 10 Hybrid Recommendations:


,Rank,Movie Title,Year,Genres,Keywords,CF Score,Content Score,Hybrid Score,Why this movie?
0,1,Mission to Mars,2000,Science Fiction,"mars, spacecraft, space, travel, alien",0.0,0.3055,0.3055,Recommended mainly from similar movie content
1,2,Aliens,1986,Horror Action Thriller Science Fiction,"android, extraterrestrial, technology, space, ...",0.0,0.2876,0.2876,Recommended mainly from similar movie content
2,3,Moonraker,1979,Action Adventure Thriller Science Fiction,"venice, mass, murder, space, marine",0.0,0.2824,0.2824,Recommended mainly from similar movie content
3,4,Alien³,1992,Science Fiction Action Horror,"prison, android, spacecraft, space, marine",0.0,0.2766,0.2766,Recommended mainly from similar movie content
4,5,Spaceballs,1987,Comedy Science Fiction,"android, lasergun, swordplay, temple, space",0.0,0.2561,0.2561,Recommended mainly from similar movie content
5,6,Lifeforce,1985,Fantasy Horror Science Fiction Thriller,"space, marine, vampire, flying, saucer",0.0,0.2549,0.2549,Recommended mainly from similar movie content
6,7,Treasure Planet,2002,Adventure Animation Family Fantasy Science Fic...,"cyborg, based, on, novel, space",0.0,0.2532,0.2532,Recommended mainly from similar movie content
7,8,Lockout,2012,Action Thriller Science Fiction,"usa, president, anti, hero, dementia",0.0,0.2523,0.2523,Recommended mainly from similar movie content
8,9,Alien,1979,Horror Action Thriller Science Fiction,"android, countdown, space, marine, space",0.0,0.2457,0.2457,Recommended mainly from similar movie content
9,10,Planet of the Apes,2001,Thriller Science Fiction Action Adventure,"gorilla, space, marine, space, suit",0.0,0.2453,0.2453,Recommended mainly from similar movie content


,Rank,Movie Title,Year,Genres,Keywords,CF Score,Content Score,Hybrid Score
0,1,Dances with Wolves,1990,Adventure Drama Western,"countryside, based, on, novel, suicide",0.9254,0.1012,0.5957
1,2,The Fifth Element,1997,Adventure Fantasy Action Thriller Science Fiction,"clone, taxi, cyborg, egypt, future",0.8376,0.1660,0.5689
2,3,Interview with the Vampire,1994,Horror Romance,"paris, san, francisco, vampire, plantation",0.9025,0.0131,0.5467
3,4,Clueless,1995,Comedy Drama Romance,"puberty, high, school, make, a",0.9079,0.0031,0.5460
4,5,The Rock,1996,Action Adventure Thriller,"san, francisco, fbi, gas, attack",0.8794,0.0340,0.5412
5,6,Alien,1979,Horror Action Thriller Science Fiction,"android, countdown, space, marine, space",0.7366,0.2457,0.5402
6,7,Independence Day,1996,Action Adventure Science Fiction,"spacecraft, patriotism, countdown, independenc...",0.7738,0.1588,0.5278
7,8,The Fugitive,1993,Adventure Action Thriller Crime Mystery,"chicago, showdown, undercover, surgeon, death",0.8759,0.0055,0.5278
8,9,Titanic,1997,Drama Romance Thriller,"shipwreck, iceberg, ship, panic, titanic",0.8681,0.0146,0.5267
9,10,X-Men,2000,Adventure Action Science Fiction,"mutant, marvel, comic, superhero, based",0.8569,0.0246,0.5240


In [22]:
# 22. Hybrid Genre Precision@10

def hybrid_genre_precision_at_k(
    title,
    k=10
):

    matched_title = find_movie_title(
        title
    )


    if matched_title is None:
        return None


    selected_genres = set(

        df.loc[
            indices[
                matched_title.lower()
            ],
            "genres_clean"
        ].split()

    ) - {""}


    if not selected_genres:
        return None


    recommended = hybrid_recommend(
        matched_title,
        top_n=k
    )


    if (
        recommended is None
        or recommended.empty
    ):

        return None


    matches = recommended[
        "Genres"
    ].fillna("").apply(

        lambda genres:

        bool(

            selected_genres
            &
            (
                set(
                    str(
                        genres
                    ).split()
                ) - {""}
            )

        )

    )


    return matches.mean()

In [23]:
# 23. Evaluate Hybrid System

random.seed(42)


valid_titles = (

    df[
        df["genres_clean"]
        .str.strip() != ""
    ]["title"]

    .drop_duplicates()

    .tolist()

)


sample_size = min(
    100,
    len(valid_titles)
)


sample_titles = random.sample(
    valid_titles,
    sample_size
)


precisions = []


for title in sample_titles:

    precision = (
        hybrid_genre_precision_at_k(
            title,
            k=10
        )
    )

    if precision is not None:

        precisions.append(
            precision
        )


if precisions:

    avg_precision = (
        sum(precisions)
        /
        len(precisions)
    )


    print(
        f"Average Hybrid "
        f"Genre Precision@10 "
        f"(n={len(precisions)}): "
        f"{avg_precision:.4f}"
    )

else:

    print(
        "No valid titles were "
        "available for evaluation."
    )

Selected Movie: The Polar Express
Selected Movie: Good Will Hunting
Selected Movie: Here On Earth
Selected Movie: Mamma Mia!
Selected Movie: The Flower of Evil
Selected Movie: The X Files: I Want to Believe
Selected Movie: The Kid
Selected Movie: Teenage Mutant Ninja Turtles: Out of the Shadows
Selected Movie: Jay and Silent Bob Strike Back
Selected Movie: Winnie the Pooh
Selected Movie: The Matrix
Selected Movie: The Lives of Others
Selected Movie: A Very Long Engagement
Selected Movie: Without a Paddle
Selected Movie: Kate & Leopold
Selected Movie: The Last Days on Mars
Selected Movie: Casino
Selected Movie: Cries and Whispers
Selected Movie: Freeheld
Selected Movie: The Lincoln Lawyer
Selected Movie: Logan's Run
Selected Movie: The Vow
Selected Movie: Quarantine
Selected Movie: Charlie and the Chocolate Factory
Selected Movie: Primary Colors
Selected Movie: I Married a Strange Person!
Selected Movie: Green Zone
Selected Movie: RockNRolla
Selected Movie: One Hour Photo
Selected Movie